In [2]:
import math

def quaternion_multiply(q1, q2):
    """
    Hamilton product of two quaternions q1 and q2.
    Each quaternion is (w, x, y, z).
    Returns q1 * q2 in the same format.
    """
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    
    w = w1*w2 - x1*x2 - y1*y2 - z1*z2
    x = w1*x2 + x1*w2 + y1*z2 - z1*y2
    y = w1*y2 + y1*w2 + z1*x2 - x1*z2
    z = w1*z2 + z1*w2 + x1*y2 - y1*x2
    return (w, x, y, z)

def rotate_quaternion_axis(q, angle_radians, axis='x'):
    """
    Rotate quaternion q = (w, x, y, z) by 'angle_radians' about
    axis 'x', 'y', or 'z'.
    
    By default, this does a 'local' rotation:
      q_new = q * r
    If you want a global/world rotation, do q_new = r * q.
    """
    half = 0.5 * angle_radians
    s = math.sin(half)
    c = math.cos(half)
    
    if axis.lower() == 'x':
        r = (c, s, 0.0, 0.0)
    elif axis.lower() == 'y':
        r = (c, 0.0, s, 0.0)
    elif axis.lower() == 'z':
        r = (c, 0.0, 0.0, s)
    else:
        raise ValueError("axis must be 'x', 'y', or 'z'")
    
    return quaternion_multiply(q, r)


# ----------------------
# Demo:
# Rotate by -90° about X, then -90° about Y (LOCAL rotations)
# ----------------------

# 1) Start with the identity quaternion (no rotation)
q_identity = (1.0, 0.0, 0.0, 0.0)

# 2) Rotate -90° about local X
# q_after_x = rotate_quaternion_axis(q_identity, math.radians(180), 'x')
# print("After -90° about local X:", q_after_x)

# # 3) Rotate another -90° about the *new/local* Y
# q_final = rotate_quaternion_axis(q_after_x, math.radians(0), 'z')
# print("After then -90° about new/local Y:", q_final)

# 2) Rotate -90° about local X
q_after_x = rotate_quaternion_axis(q_identity, math.radians(180), 'x')
print("After -90° about local X:", q_after_x)

# 3) Rotate another -90° about the *new/local* Y
q_final = rotate_quaternion_axis(q_after_x, math.radians(0), 'z')
print("After then -90° about new/local Y:", q_final)


After -90° about local X: (6.123233995736766e-17, 1.0, 0.0, 0.0)
After then -90° about new/local Y: (6.123233995736766e-17, 1.0, 0.0, 0.0)


In [ ]:
import jax.numpy as jnp
import math 

def quaternion_multiply(q1: jnp.ndarray, q2: jnp.ndarray) -> jnp.ndarray:
    """
    Hamilton product of two quaternions q1 and q2.
    Each quaternion is (w, x, y, z).
    Returns q1 * q2 in the same format as a jax.numpy array.
    """
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    
    w = w1*w2 - x1*x2 - y1*y2 - z1*z2
    x = w1*x2 + x1*w2 + y1*z2 - z1*y2
    y = w1*y2 + y1*w2 + z1*x2 - x1*z2
    z = w1*z2 + z1*w2 + x1*y2 - y1*x2
    return jnp.array([w, x, y, z])


def rotate_quaternion_axis(q: jnp.ndarray, angle_radians: float, axis: str = 'x') -> jnp.ndarray:
    """
    Rotates quaternion q = (w, x, y, z) by 'angle_radians' about axis ('x', 'y', or 'z').
    Returns q * r as a jax.numpy array.
    """
    half = 0.5 * angle_radians
    s = jnp.sin(half)
    c = jnp.cos(half)
    
    if axis.lower() == 'x':
        r = jnp.array([c, s, 0.0, 0.0])
    elif axis.lower() == 'y':
        r = jnp.array([c, 0.0, s, 0.0])
    elif axis.lower() == 'z':
        r = jnp.array([c, 0.0, 0.0, s])
    else:
        raise ValueError("axis must be 'x', 'y', or 'z'")
    
    return quaternion_multiply(q, r)

def quaternion_inverse(q: jnp.ndarray) -> jnp.ndarray:
    """
    Returns the inverse of a normalized quaternion q.
    """
    return jnp.array([q[0], -q[1], -q[2], -q[3]])

def update_orientation(initial_quat, rotate_x, rotate_z):
    """Rotate an orientation by rotate_x about x-axis and rotate_z about z-axis."""
    qx = rotate_quaternion_axis(initial_quat, math.radians(rotate_x), 'x')
    return rotate_quaternion_axis(qx, math.radians(rotate_z), 'z')


def smooth_orientation_update(current_eef_orientation, desired_orientation, gain=0.1):
    """
    Rotates current orientation a fraction 'gain' of the way toward desired_orientation.
    """
    # Ensure they're normalized
    cur = current_eef_orientation / jnp.linalg.norm(current_eef_orientation)
    des = desired_orientation / jnp.linalg.norm(desired_orientation)

    # difference quaternion: q_diff = des * inv(cur)
    q_diff = quaternion_multiply(des, quaternion_inverse(cur))

    # angle of difference
    angle_diff = 2.0 * jnp.arccos(jnp.clip(q_diff[0], -1.0, 1.0))

    # small step angle
    step_angle = gain * angle_diff

    # axis of rotation
    small_eps = 1e-8
    sin_half = jnp.sin(step_angle / 2.0)
    axis = q_diff[1:]
    axis_norm = jnp.linalg.norm(axis)

    axis = jnp.where(axis_norm > small_eps, axis / axis_norm, jnp.array([0.0, 0.0, 1.0]))

    # quaternion for the incremental step
    incremental_rotation = jnp.concatenate([
        jnp.array([jnp.cos(step_angle/2.0)]),
        axis * sin_half
    ])

    # new orientation = incremental step * current
    new_orientation = quaternion_multiply(incremental_rotation, cur)
    return new_orientation / jnp.linalg.norm(new_orientation)

gain_yaw = 0.5
target_rot = jnp.array([1.0, 0.0, 0.0, 0.0])
desired_orientation = update_orientation(target_rot, rotate_x = 180.0, rotate_z = 10.0)
# smooth_orientation = smooth_orientation_update(current_eef_orientation, 
#                                                 desired_orientation,
#                                                 gain=gain_yaw)

In [4]:
desired_orientation

Array([-4.3545054e-08,  9.9619472e-01,  8.7155737e-02,  3.8096983e-09],      dtype=float32)

In [5]:
target_rot

Array([1., 0., 0., 0.], dtype=float32)

In [9]:
desired_orientation

Array([ 0.64034164,  0.7631294 , -0.06676517,  0.05602263], dtype=float32)

In [7]:
from typing import Union, List
import numpy as np
from scipy.spatial.transform import Rotation as R

def compute_end_effector_quaternion(
    cube_quat: Union[List[float], np.ndarray]
) -> np.ndarray:
    """
    Compute the end-effector quaternion such that the cube's local axes align as follows:
    cube x-axis → world -z
    cube y-axis → world  x
    cube z-axis → world  y

    Parameters:
    - cube_quat: Quaternion of the cube [w, x, y, z]

    Returns:
    - end_effector_quat: Quaternion for end-effector [w, x, y, z]
    """
    # Reorder from [w, x, y, z] to [x, y, z, w]
    cube_quat_xyzw = np.array([cube_quat[1], cube_quat[2], cube_quat[3], cube_quat[0]])
    q_cube = R.from_quat(cube_quat_xyzw)

    # Rotation to align cube's local axes to desired world axes
    R_align = np.array([
        [ 1,  0,  0],   # new x-axis (was y)
        [ 0, -1,  0],   # new y-axis (was z)
        [ 0,  0, -1]    # new z-axis (was x)
    ])
    q_align = R.from_matrix(R_align)

    # Compute final end-effector quaternion
    q_ee = q_cube * q_align
    q_ee_xyzw = q_ee.as_quat()  # [x, y, z, w]

    # Reorder back to [w, x, y, z]
    q_ee_wxyz = np.array([q_ee_xyzw[3], q_ee_xyzw[0], q_ee_xyzw[1], q_ee_xyzw[2]])
    return q_ee_wxyz


In [8]:
compute_end_effector_quaternion([1,0,0,0])

array([0., 1., 0., 0.])

In [4]:
import numpy as np
from scipy.spatial.transform import Rotation as R
from typing import Union, List
from itertools import permutations, product

def generate_valid_rotation_matrices() -> List[np.ndarray]:
    """
    Generate all 3x3 proper rotation matrices from permutations and sign flips
    of the identity matrix axes.
    """
    base_axes = np.eye(3)
    matrices = []

    for perm in permutations([0, 1, 2]):
        for signs in product([-1, 1], repeat=3):
            R_candidate = np.column_stack([signs[i] * base_axes[:, perm[i]] for i in range(3)])
            if np.linalg.det(R_candidate) > 0:  # Ensure right-handed
                matrices.append(R_candidate)
    return matrices

def compute_all_end_effector_quaternions(
    cube_quat: Union[List[float], np.ndarray]
) -> List[np.ndarray]:
    """
    Compute all possible end-effector quaternions to align the cube with
    different valid world alignments (based on proper rotation matrices).

    Parameters:
    - cube_quat: Quaternion of the cube [x, y, z, w]

    Returns:
    - List of end-effector quaternions [x, y, z, w] for each valid alignment
    """
    q_cube = R.from_quat(cube_quat)
    valid_rot_matrices = generate_valid_rotation_matrices()

    ee_quaternions = []
    for R_align in valid_rot_matrices:
        q_align = R.from_matrix(R_align)
        q_ee = q_cube * q_align
        ee_quaternions.append(q_ee.as_quat())
    
    return ee_quaternions


In [ ]:
cube_quat = [1,0,0,0]  # Identity quaternion (no rotation)
quats = compute_all_end_effector_quaternions(cube_quat)

for i, quat in enumerate(quats):
    print(f"Rotation {i+1}: {quat}")
